In [14]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()

model = "claude-sonnet-4-5"


### Making my first request

message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

print(message.content[0].text)


### Multi- Turn Conversations 
### Building helper functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

# putting into practice 

messages = []

add_user_message(messages, "Define quantum computing in one sentence")

answer = chat(messages)

add_assistant_message(messages, answer)

add_user_message(messages, "Write Another Sentence")

final_answer = chat(messages)



### Lesson 5 System prompts
# building a flexible cha fucntion

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }


    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

## Now you can call the chat function with or without a system promt:

# Without system prompt
answer = chat(messages)

# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""
answer = chat(messages, system=system)

print(answer)




Quantum computing is a type of computation that harnesses quantum mechanical phenomena like superposition and entanglement to process information in ways that can solve certain problems exponentially faster than classical computers.
Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, enabling them to explore many possible solutions to a problem at once.


### Prompting a eval workflow 

In [15]:
# This prompt will be the baseline for testing and improvement 

prompt1 = f"""
Please answer the user's question:

{"Whats 2+2?"}
"""

In [16]:
prompt2 = f"""
Please answer the user's question:

{"Whats 2+2?"}

Answer the question with ample detail
"""

In [17]:
messages = []

add_user_message(messages, prompt2)
chat(messages, system=None)

'# What\'s 2+2?\n\nThe answer is **4**.\n\n## Detailed Explanation\n\n### Basic Addition\n2 + 2 = 4 is one of the most fundamental arithmetic operations in mathematics. When you combine two units with another two units, you get four units total.\n\n### Visual Representation\nYou can think of it in several ways:\n- **Counting objects**: If you have 2 apples and someone gives you 2 more apples, you now have 4 apples\n- **Number line**: Starting at 2 on a number line and moving 2 spaces to the right lands you on 4\n- **Tally marks**: || + || = ||||\n\n### Mathematical Context\nThis operation demonstrates:\n- **Commutative property**: 2 + 2 = 2 + 2 (order doesn\'t matter)\n- **Even number addition**: Adding two even numbers always results in an even number\n- **Doubling**: 2 + 2 is the same as 2 × 2, both equal 4\n\n### Historical Significance\nThe equation "2+2=4" is often used in literature and philosophy as an example of an objective, undeniable truth. George Orwell famously referenced 

### Generating Test datsets 

First we need our helper functions

In [18]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
import json

client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [19]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def chat(messages, system=None, stop_sequences=None, max_tokens=4000):
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    if message.stop_reason == "max_tokens":
        print("WARNING: truncated")
    return message.content[0].text

In [20]:
def generate_dataset(n=3):
    prompt = f"""
Generate an evaluation dataset for a prompt evaluation. The dataset will be used
to evaluate prompts that generate Python, JSON, or Regex specifically for
AWS-related tasks. Generate an array of JSON objects, each representing a task
that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {{"task": "Description of task"}}
]
```

* Focus on tasks solvable by a single Python function, JSON object, or regex
* Focus on tasks that do not require writing much code

Please generate {n} objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text.strip())

In [21]:
dataset = generate_dataset(3)

for item in dataset:
    print("-", item["task"])

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

print(f"\nSaved {len(dataset)} tasks")

- Create a JSON configuration object for an AWS S3 bucket policy that allows public read access to all objects
- Write a Python function that parses an AWS CloudWatch log entry and extracts the timestamp, log level, and message using regex
- Create a JSON object representing an AWS Lambda function environment variables configuration with database credentials and API endpoints

Saved 3 tasks


### Running the Eval

In [22]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [23]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [24]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [25]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [26]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Policy - Public Read Access\n\nHere's a JSON configuration object for an S3 bucket policy that allows public read access:\n\n```json\n{\n  \"Version\": \"2012-10-17\",\n  \"Statement\": [\n    {\n      \"Sid\": \"PublicReadGetObject\",\n      \"Effect\": \"Allow\",\n      \"Principal\": \"*\",\n      \"Action\": \"s3:GetObject\",\n      \"Resource\": \"arn:aws:s3:::your-bucket-name/*\"\n    }\n  ]\n}\n```\n\n## Key Components\n\n| Component | Description |\n|-----------|-------------|\n| **Version** | Policy language version (always \"2012-10-17\") |\n| **Statement** | Array containing policy rules |\n| **Sid** | Statement identifier (optional but recommended) |\n| **Effect** | \"Allow\" or \"Deny\" |\n| **Principal** | \"*\" means anyone (public access) |\n| **Action** | \"s3:GetObject\" allows read access |\n| **Resource** | ARN specifying the bucket and objects |\n\n## Important Notes\n\n\u26a0\ufe0f **Security Warning**: This policy makes your e

### Model Based GRADING

In [27]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case["task"]}
Solution: {output}

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10
"""
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text.strip())

In [28]:
results = []

for case in dataset:
    try:
        solution = chat([{"role": "user", "content": case["task"]}])
        grade = grade_by_model(case, solution)
        results.append({
            "task": case["task"],
            "solution": solution,
            "score": grade["score"],
            "strengths": grade["strengths"],
            "weaknesses": grade["weaknesses"],
        })
        print(f"{grade['score']}/10 — {case['task'][:60]}")
    except Exception as e:
        print(f"FAILED — {case['task'][:60]}: {e}")

7/10 — Create a JSON configuration object for an AWS S3 bucket poli
6/10 — Write a Python function that parses an AWS CloudWatch log en
FAILED — Create a JSON object representing an AWS Lambda function env: 'score'


#### Code Based GRading

In [29]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

In [33]:
{
    "task": "Create a Python function to validate anAWS IAM username",
    "format": "python"
}

def run_prompt(test_case):
    prompt = f"""
Please provide a solution to the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")

    return chat(messages, stop_sequences=["```"])


In [34]:
add_assistant_message(messages, "```code")

In [36]:
def grade_output(test_case, output):
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    syntax_score = grade_syntax(output, test_case)

    return {
        "score": (model_score + syntax_score) / 2,
        "model_score": model_score,
        "syntax_score": syntax_score,
        "strengths": model_grade["strengths"],
        "weaknesses": model_grade["weaknesses"],
        "reasoning": model_grade["reasoning"],
    }

In [37]:
results = []

for case in dataset:
    try:
        output = run_prompt(case)
        grade = grade_output(case, output)
        results.append({"task": case["task"], "output": output, **grade})
        print(f"{grade['score']:.1f}/10  (model {grade['model_score']}, syntax {grade['syntax_score']})  {case['task'][:50]}")
    except Exception as e:
        print(f"FAILED  {case['task'][:50]}: {e}")

scores = [r["score"] for r in results]
print(f"\nBaseline: {sum(scores)/len(scores):.2f} over {len(results)} cases")

FAILED  Create a JSON configuration object for an AWS S3 b: name 'grade_syntax' is not defined
FAILED  Write a Python function that parses an AWS CloudWa: name 'grade_syntax' is not defined
FAILED  Create a JSON object representing an AWS Lambda fu: name 'grade_syntax' is not defined


ZeroDivisionError: division by zero